In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('CleanData.csv')

df.head()

,run_id,is_correct,response,rt,task_part,time_elapsed,trial_index,Honestly,Complexity,recorded_at
0,158,1.0,2,332634,decision,433571,3,Honest,Complex,2026-06-12 20:42:44
1,137,1.0,2,275009,decision,1936602,3,Honest,Complex,2026-06-11 14:40:39
2,165,0.0,3,266566,decision,267557,1,Dishonest,Complex,2026-06-13 18:06:08
3,30,0.0,0,200735,decision,261058,3,Honest,Complex,2026-06-08 23:12:19
4,30,0.0,2,200222,decision,466742,5,Dishonest,Simple,2026-06-08 23:12:19


In [2]:
decisions_df = df[df['task_part'] == 'decision'].copy()

In [3]:
confidence_df = df[df['task_part'] == 'confidence'].copy()

In [4]:
if not confidence_df.empty:
    # Convert response to numeric confidence score
    confidence_df['confidence'] = pd.to_numeric(confidence_df['response'], errors='coerce')
    
    confidence_df['decision_trial_index'] = confidence_df['trial_index'] - 1
    
    # Merge using 'decision_trial_index' matching the decisions_df 'trial_index'
    analysis_df = pd.merge(
        decisions_df, 
        confidence_df[['run_id', 'decision_trial_index', 'confidence']], 
        left_on=['run_id', 'trial_index'], 
        right_on=['run_id', 'decision_trial_index'],
        how='left'
    )
    
    # Clean up the extra join column
    analysis_df = analysis_df.drop(columns=['decision_trial_index'])
    analysis_df = analysis_df.rename(columns={"Honestly": "Honesty"})
    has_confidence = True

In [5]:
groupby_cols = ['Complexity']
agg_dict = {
    'is_correct': 'mean',
    'rt': 'mean'
}
if has_confidence:
    agg_dict['confidence'] = 'mean'

complexity_summary = analysis_df.groupby(groupby_cols).agg(agg_dict).rename(
    columns={'is_correct': 'Accuracy (Mean)', 'rt': 'Reaction Time (ms)', 'confidence': 'Confidence (Mean)'}
)
print(complexity_summary)

            Accuracy (Mean)  Reaction Time (ms)  Confidence (Mean)
Complexity                                                        
Complex            0.271523        47156.596026          59.735099
Simple             0.442308        34755.102564          76.698718


In [6]:
analysis_df.head()

,run_id,is_correct,response,rt,task_part,time_elapsed,trial_index,Honesty,Complexity,recorded_at,confidence
0,158,1.0,2,332634,decision,433571,3,Honest,Complex,2026-06-12 20:42:44,90
1,137,1.0,2,275009,decision,1936602,3,Honest,Complex,2026-06-11 14:40:39,30
2,165,0.0,3,266566,decision,267557,1,Dishonest,Complex,2026-06-13 18:06:08,75
3,30,0.0,0,200735,decision,261058,3,Honest,Complex,2026-06-08 23:12:19,45
4,30,0.0,2,200222,decision,466742,5,Dishonest,Simple,2026-06-08 23:12:19,85


In [7]:
analysis_df['is_correct'] = analysis_df['is_correct'].astype(int)

print("## 1. SPEED (Reaction Time) - Linear Mixed Model ##")

# Formula: rt ~ Complexity * honestly
# Random effect: groups='run_id' (random intercept for each subject)

lmm_speed = smf.mixedlm("rt ~ Complexity * Honesty", analysis_df, groups=analysis_df["run_id"])
lmm_speed_results = lmm_speed.fit()

print(lmm_speed_results.summary())
print("-" * 80)


print("\n## 2. CONFIDENCE - Linear Mixed Model ##")
    
lmm_conf = smf.mixedlm("confidence ~ Complexity * Honesty", analysis_df, groups=analysis_df["run_id"])
lmm_conf_results = lmm_conf.fit()
    
print(lmm_conf_results.summary())

print("-" * 80)

# Family: Binomial (since data is binary 0/1)
# Link function: Logit (Logistic regression with mixed effects)

glmm_acc = smf.gee("is_correct ~ Complexity * Honesty", 
                   groups="run_id", 
                   data=analysis_df, 
                   family=sm.families.Binomial())
glmm_acc_results = glmm_acc.fit()

print(glmm_acc_results.summary())

## 1. SPEED (Reaction Time) - Linear Mixed Model ##
                             Mixed Linear Model Regression Results
Model:                        MixedLM            Dependent Variable:            rt             
No. Observations:             307                Method:                        REML           
No. Groups:                   103                Scale:                         1339687349.6078
Min. group size:              1                  Log-Likelihood:                -3651.8235     
Max. group size:              3                  Converged:                     Yes            
Mean group size:              3.0                                                              
-----------------------------------------------------------------------------------------------
                                           Coef.     Std.Err.   z    P>|z|   [0.025     0.975] 
-----------------------------------------------------------------------------------------------
Intercept        